<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/healthstack-logo-dark.png" alt="HealthStack logo" width="320" />
</div>

<p align="center"><strong>AFROTECH 2026</strong></p>
<p align="center"><font size="6"><strong>Building Fairness Into the Data Supply Chain</strong></font></p>
<p align="center"><strong>Instructor: Waverly Rose Brim</strong></p>

---

<p align="center"><font size="6"><strong>The Second Failure</strong></font></p>

<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/second-failure.jpg" alt="A blister pack of US quarters with empty pockets beside loose quarters scattered on a dark background." width="640" />
  <p><em>A global number can look complete while a group is missing.</em></p>
</div>

---

## Start here
This notebook answers three questions:
1. Does the same model look different when you slice it by insurance?
2. Which ship-gate rule fails, and why?
3. Why does the slice you choose decide what you can find?

**Workflow:** predict -> run -> inspect -> interpret.

## Colab setup
Skip this cell if you are running locally.

In [ ]:
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps", "--no-cache-dir",
        "data-supply-chain-workshop==0.1.2",
    ])
else:
    import data_supply_chain

## Cold start
Same cohort, same split, whichever notebook you opened first. Earlier
notebooks do not need to be open.

In [ ]:
from data_supply_chain.audit import subgroup_audit
from data_supply_chain.bootstrap import cold_start
from data_supply_chain.models import fit_audit_model

S = fit_audit_model(cold_start(), "egfr_2021")
print(f"{len(S.df):,} adults · source={S.source} · real={S.is_real} · global AUROC={S.auroc:.3f}")

By race, the harm sat at the decision threshold and AUROC could not see it.
Now slice the same model a different way.

In [ ]:
from data_supply_chain.audit import ship_gate

ins_tab, ins_summ = subgroup_audit(S.audit_sub, S.y_test, S.predictions, "insurance", n_boot=500)
print(ins_tab.round(3).to_string(index=False))
print(f"\nglobal AUROC = {S.auroc:.3f}")

### **Attendee challenge: which rule fails?**

The table is the audit. The gate has three rules: AUROC floor, ECE cap, and
maximum drop from global.

**Floor, calibration, or drop from global?**

Do not run the gate cell until you have an answer.

In [ ]:
# Your turn — which rule fails?

import inspect

_rules = inspect.signature(ship_gate).parameters
floor = _rules["min_worst_auroc"].default
max_ece = _rules["max_worst_ece"].default
max_drop = _rules["max_drop"].default

worst = ins_tab.loc[ins_tab["auroc"].idxmin()]
drop = S.auroc - worst["auroc"]

print(f"Rules: AUROC floor ≥ {floor:.2f}, ECE ≤ {max_ece:.2f}, drop from global ≤ {max_drop:.2f}\n")
print(f"Worst group: {worst['group']}")
print(f"AUROC: {worst['auroc']:.3f}")
print(f"ECE: {worst['ece']:.3f}")
print(f"Drop from global: {drop:.3f}")

In [ ]:
gate_verdict, gate_reasons = ship_gate(
    S.auroc, ins_summ["worst_auroc"], ins_summ["worst_ece"],
)
print(f"global AUROC={S.auroc:.3f}  ->  MODEL GATE (insurance): {gate_verdict}")
print(f"   {gate_reasons if gate_reasons else '(all floors met)'}")

In [ ]:
# Demo, take this opportunity to review the output

from IPython.display import Image, SVG, display
from pathlib import Path
import tempfile

from data_supply_chain import figures as _fig

def _show_workshop_fig(png_name, plot_name, *args):
    """PNG helpers shipped after 0.1.1. Colab still has 0.1.1; fall back to SVG."""
    png = getattr(_fig, png_name, None)
    if png is not None:
        display(Image(data=png(*args), format="png"))
        return
    out = Path(tempfile.mkdtemp()) / "fig.svg"
    getattr(_fig, plot_name)(*args, out)
    display(SVG(filename=str(out)))

In [ ]:
_show_workshop_fig(
    "audit_bar_png", "plot_audit_bar",
    ins_tab, S.auroc, "Act 2c: worst-group audit by insurance",
)

### **Attendee challenge: change only the slice**

Same model. Same predictions. Change one word.

Run this with `race`, then change it to `insurance`.

Before you run the second one: **do you expect the gate verdict to change?**

In [ ]:
# Your turn — change only the slice

slice_by = "race"  # then "insurance"

tab, summary = subgroup_audit(
    S.audit_sub, S.y_test, S.predictions, slice_by, n_boot=100,
)
verdict, reasons = ship_gate(
    S.auroc, summary["worst_auroc"], summary["worst_ece"],
)

print(f"Slice: {slice_by}")
print(f"Worst-group AUROC: {summary['worst_auroc']:.3f}")
print(f"Drop from global: {S.auroc - summary['worst_auroc']:.3f}")
print(f"Gate: {verdict}")
print(f"Reason: {reasons or 'all rules passed'}")

## Two failures, two hiding places
By **race**, the harm sat at the decision threshold, where AUROC is not built
to look. By **insurance**, the model's discrimination is measurably lower, and
the global number hides it until you slice. One metric was blind; one was hiding.

We only found the insurance failure because someone thought to name insurance.
That is what step 9 is for.

---

## Maintainer cell (safe to skip in workshop)
This writes the findings the website reads, so the numbers on the page stay
synced to this notebook's output.

In [ ]:
import inspect

from data_supply_chain.findings import Findings
from data_supply_chain.paths import findings_path

f = Findings.open()
for _row in ins_tab.itertuples():
    _key = str(_row.group).lower()
    f.record(f"act2.insurance.{_key}.n", int(_row.n))
    f.record(f"act2.insurance.{_key}.auroc", round(float(_row.auroc), 3))
    f.record(f"act2.insurance.{_key}.lo", round(float(_row.lo), 3))
    f.record(f"act2.insurance.{_key}.hi", round(float(_row.hi), 3))
    f.record(f"act2.insurance.{_key}.ece", round(float(_row.ece), 3))

f.record("act2.gate.verdict", gate_verdict)
f.record("act2.gate.reason", gate_reasons[0] if gate_reasons else "")
f.record("act2.gate.drop", round(S.auroc - ins_summ["worst_auroc"], 3))

_rules = inspect.signature(ship_gate).parameters
f.record("act2.gate.floor", _rules["min_worst_auroc"].default)
f.record("act2.gate.max_drop", _rules["max_drop"].default)
f.record("act2.gate.max_ece", _rules["max_worst_ece"].default)

if findings_path() is None:
    print("Outside a repo checkout - findings.json not written (expected in Colab).")
else:
    f.write(state=S)
    print("wrote findings.json")